# RNNs and LSTMs — Implementations

Each cell twice more: once on tensors with autograd replacing the hand-written BPTT (agreeing with it validates the notebook's hardest derivation), and once through `torch.nn` with the scratch weights copied in — which for the LSTM means the classic i,f,g,o vs f,i,g,o gate-order translation. Every lane draws its weights from the same NumPy generator and pushes the same fixed upstream gradient, so states and gradients must match to machine precision.

## 15_rnn_cell

One tanh recurrence, unrolled.

### torch

The same recurrence on float64 tensors, weights drawn from the same NumPy generator. **What torch adds:** autograd — `backward` becomes one call on `(h_all · dh_all).sum()` and must reproduce the hand-derived BPTT gradients exactly.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw W_xh first, then W_hh, from the numpy rng — same order as the scratch init.
# 2. as_tensor keeps float64; requires_grad_(True) makes the weights autograd leaves.
# 3. Collect every h_t in a list and torch.stack it — the stack stays on the graph.
# 4. backward is (h_all * dh_all).sum().backward(); BPTT is just the chain rule.
# 5. Set p.grad = None before backward, or autograd accumulates across calls.


def sigmoid(z):
    """Kept for step alignment with the scratch lane; torch's own is already stable."""
    return torch.sigmoid(torch.as_tensor(z))


class RNNCell:
    """Vanilla RNN cell, h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h), with autograd
    playing the role of the hand-written BPTT."""

    def __init__(self, input_dim, hidden_dim, rng):
        scale = np.sqrt(2.0 / (input_dim + hidden_dim))
        self.W_xh = torch.as_tensor(rng.normal(0, scale, (hidden_dim, input_dim))).requires_grad_(True)
        self.W_hh = torch.as_tensor(rng.normal(0, scale, (hidden_dim, hidden_dim))).requires_grad_(True)
        self.b_h = torch.zeros(hidden_dim, dtype=torch.float64, requires_grad=True)
        self.hidden_dim = hidden_dim

    def forward(self, X):
        """Roll the recurrence over the whole (seq_len, input_dim) sequence."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        h = torch.zeros(self.hidden_dim, dtype=torch.float64)
        self.cache = {'h': [h.detach().clone()]}
        h_list = []
        for t in range(Xt.shape[0]):
            h = torch.tanh(self.W_xh @ Xt[t] + self.W_hh @ h + self.b_h)
            self.cache['h'].append(h.detach().clone())
            h_list.append(h)
        self.h_all_ = torch.stack(h_list)
        return self.h_all_

    def backward(self, dh_all):
        """BPTT via autograd: differentiate <dh_all, h_all> w.r.t. the parameters."""
        dh = torch.as_tensor(np.asarray(dh_all, dtype=float))
        for p in (self.W_xh, self.W_hh, self.b_h):
            p.grad = None
        (self.h_all_ * dh).sum().backward()
        self.dW_xh = self.W_xh.grad
        self.dW_hh = self.W_hh.grad
        self.db_h = self.b_h.grad


In [ ]:
# exports: h_last, dW_xh, dW_hh, db_h
_rng_eq = np.random.default_rng(715)
_cell_eq = RNNCell(input_dim=3, hidden_dim=5, rng=_rng_eq)
X_eq = _rng_eq.normal(size=(8, 3))
h_all_eq = _cell_eq.forward(X_eq)
_dh_eq = np.linspace(-1.0, 1.0, 40).reshape(8, 5)
_cell_eq.backward(_dh_eq)
h_last = h_all_eq[-1].detach()
dW_xh = _cell_eq.dW_xh
dW_hh = _cell_eq.dW_hh
db_h = _cell_eq.db_h
print("h_last:", np.round(h_last.numpy(), 5))
print("|dW_hh| =", float(torch.linalg.norm(dW_hh)))


In [ ]:
# tanh keeps every hidden state strictly inside (-1, 1).
assert float(h_all_eq.detach().abs().max()) < 1.0, "tanh output must stay in (-1, 1)"

# Autograd agrees with a central finite difference along b_h (all-ones direction).
_eps = 1e-6
with torch.no_grad():
    _cell_eq.b_h += _eps
_lp = float((_cell_eq.forward(X_eq) * torch.as_tensor(_dh_eq)).sum())
with torch.no_grad():
    _cell_eq.b_h -= 2 * _eps
_lm = float((_cell_eq.forward(X_eq) * torch.as_tensor(_dh_eq)).sum())
with torch.no_grad():
    _cell_eq.b_h += _eps
assert abs((_lp - _lm) / (2 * _eps) - float(db_h.sum())) < 1e-5, \
    "BPTT gradient must match finite differences"

# Zero upstream gradient means zero parameter gradients.
_cell_eq.forward(X_eq)
_cell_eq.backward(np.zeros((8, 5)))
assert float(_cell_eq.dW_hh.abs().max()) == 0.0, "no signal in, no gradient out"


### library

`nn.RNNCell` is exactly `tanh(W_ih x + b_ih + W_hh h + b_hh)` — the scratch cell with its single bias split in two. **What the library adds:** the fused step and a redundant second bias; with the scratch weights copied in, states and gradients agree to the last bit.

In [ ]:
import numpy as np
import torch

# hints:
# 1. nn.RNNCell is tanh(W_ih x + b_ih + W_hh h + b_hh) — two biases where scratch has one.
# 2. Copy W_xh into weight_ih and W_hh into weight_hh; .double() the module first.
# 3. Zero both module biases; the scratch b_h starts at zero anyway.
# 4. Feed a (1, input_dim) batch each step and squeeze the batch dim back off.
# 5. bias_ih.grad equals bias_hh.grad — the split bias is redundant; db_h is either one.


class RNNCell:
    """The scratch cell driven through torch.nn.RNNCell with the same weights,
    so forward states and BPTT gradients must agree to machine precision."""

    def __init__(self, input_dim, hidden_dim, rng):
        scale = np.sqrt(2.0 / (input_dim + hidden_dim))
        self.W_xh = torch.as_tensor(rng.normal(0, scale, (hidden_dim, input_dim)))
        self.W_hh = torch.as_tensor(rng.normal(0, scale, (hidden_dim, hidden_dim)))
        self.cell = torch.nn.RNNCell(input_dim, hidden_dim, nonlinearity='tanh').double()
        with torch.no_grad():
            self.cell.weight_ih.copy_(self.W_xh)
            self.cell.weight_hh.copy_(self.W_hh)
            self.cell.bias_ih.zero_()
            self.cell.bias_hh.zero_()
        self.hidden_dim = hidden_dim

    def forward(self, X):
        """Same recurrence, one nn.RNNCell call per time step (batch of 1)."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        h = torch.zeros(1, self.hidden_dim, dtype=torch.float64)
        self.cache = {'h': [h[0].detach().clone()]}
        h_list = []
        for t in range(Xt.shape[0]):
            h = self.cell(Xt[t].unsqueeze(0), h)
            self.cache['h'].append(h[0].detach().clone())
            h_list.append(h[0])
        self.h_all_ = torch.stack(h_list)
        return self.h_all_

    def backward(self, dh_all):
        """Autograd through the module, then read the grads back in scratch names."""
        dh = torch.as_tensor(np.asarray(dh_all, dtype=float))
        self.cell.zero_grad(set_to_none=True)
        (self.h_all_ * dh).sum().backward()
        self.dW_xh = self.cell.weight_ih.grad
        self.dW_hh = self.cell.weight_hh.grad
        self.db_h = self.cell.bias_ih.grad


In [ ]:
# exports: h_last, dW_xh, dW_hh, db_h
_rng_eq = np.random.default_rng(715)
_cell_eq = RNNCell(input_dim=3, hidden_dim=5, rng=_rng_eq)
X_eq = _rng_eq.normal(size=(8, 3))
h_all_eq = _cell_eq.forward(X_eq)
_dh_eq = np.linspace(-1.0, 1.0, 40).reshape(8, 5)
_cell_eq.backward(_dh_eq)
h_last = h_all_eq[-1].detach()
dW_xh = _cell_eq.dW_xh
dW_hh = _cell_eq.dW_hh
db_h = _cell_eq.db_h
print("h_last:", np.round(h_last.numpy(), 5))
print("|dW_hh| =", float(torch.linalg.norm(dW_hh)))


In [ ]:
# With h_0 = 0 and zero biases the first state is tanh(W_xh @ x_0) — translation exact.
_h1 = np.tanh(_cell_eq.W_xh.numpy() @ X_eq[0])
assert np.allclose(h_all_eq[0].detach().numpy(), _h1, atol=1e-12), "weight copy must be exact"

# The module really carries the scratch draw, not torch's default init.
assert torch.equal(_cell_eq.cell.weight_hh.detach(), _cell_eq.W_hh)

# The split bias is redundant: both halves receive the identical gradient.
assert torch.allclose(_cell_eq.cell.bias_ih.grad, _cell_eq.cell.bias_hh.grad, atol=1e-12), \
    "b_ih and b_hh sit in the same pre-activation, so their grads coincide"


## 15_lstm_cell

Gated memory with a gradient highway.

### torch

The notebook's LSTM on tensors: same stacked `W` in f,i,g,o order, same forget-bias of 1. **What torch adds:** autograd walks the unrolled graph, so the whole `backward` — the trickiest code in the topic — collapses to one line, and matching the hand BPTT validates both.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Same layout as the notebook: rows of W are f, i, g, o and concat is [h, x].
# 2. One rng.normal draw for the whole (4m, m+d) W; forget bias b[:m] = 1.0, rest zero.
# 3. c = f*c + i*g, h = o*tanh(c); cache detached copies, keep the live graph on h.
# 4. Autograd replaces the hand BPTT: (h_all * dh_all).sum().backward() fills W.grad.


class LSTMCell:
    """LSTM with the notebook's conventions (stacked W in f,i,g,o order, forget
    bias of 1) and autograd standing in for the hand-derived backward pass."""

    def __init__(self, input_dim, hidden_dim, rng):
        self.hidden_dim = hidden_dim
        concat_dim = hidden_dim + input_dim
        scale = np.sqrt(2.0 / concat_dim)
        self.W = torch.as_tensor(rng.normal(0, scale, (4 * hidden_dim, concat_dim))).requires_grad_(True)
        b = np.zeros(4 * hidden_dim)
        b[:hidden_dim] = 1.0
        self.b = torch.as_tensor(b).requires_grad_(True)

    def forward(self, X):
        """Roll gates, cell state and hidden state over the sequence."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        m = self.hidden_dim
        h = torch.zeros(m, dtype=torch.float64)
        c = torch.zeros(m, dtype=torch.float64)
        self.cache = {'h': [h.detach().clone()], 'c': [c.detach().clone()]}
        h_list = []
        for t in range(Xt.shape[0]):
            gates = self.W @ torch.cat([h, Xt[t]]) + self.b
            f = torch.sigmoid(gates[:m])
            i = torch.sigmoid(gates[m:2 * m])
            g = torch.tanh(gates[2 * m:3 * m])
            o = torch.sigmoid(gates[3 * m:])
            c = f * c + i * g
            h = o * torch.tanh(c)
            self.cache['c'].append(c.detach().clone())
            self.cache['h'].append(h.detach().clone())
            h_list.append(h)
        self.h_all_ = torch.stack(h_list)
        return self.h_all_

    def backward(self, dh_all):
        """The whole BPTT — the trickiest code in the topic — as one autograd call."""
        dh = torch.as_tensor(np.asarray(dh_all, dtype=float))
        for p in (self.W, self.b):
            p.grad = None
        (self.h_all_ * dh).sum().backward()
        self.dW = self.W.grad
        self.db = self.b.grad


In [ ]:
# exports: h_last, c_last, dW, db
_rng_eq = np.random.default_rng(716)
_cell_eq = LSTMCell(input_dim=3, hidden_dim=5, rng=_rng_eq)
X_eq = _rng_eq.normal(size=(8, 3))
h_all_eq = _cell_eq.forward(X_eq)
_dh_eq = np.linspace(-1.0, 1.0, 40).reshape(8, 5)
_cell_eq.backward(_dh_eq)
h_last = h_all_eq[-1].detach()
c_last = _cell_eq.cache['c'][-1]
dW = _cell_eq.dW
db = _cell_eq.db
print("h_last:", np.round(h_last.numpy(), 5))
print("|dW| =", float(torch.linalg.norm(dW)))


In [ ]:
# h = o * tanh(c) is bounded even though the cell state c is not.
assert float(h_all_eq.detach().abs().max()) < 1.0, "hidden state must stay in (-1, 1)"

# Recompute step 0 by hand with the notebook's f,i,g,o slicing (c_0 = 0, h_0 = 0).
_W0 = _cell_eq.W.detach().numpy(); _b0 = _cell_eq.b.detach().numpy(); _m = 5
_g0 = _W0 @ np.concatenate([np.zeros(_m), X_eq[0]]) + _b0
_f0 = 1 / (1 + np.exp(-_g0[:_m])); _i0 = 1 / (1 + np.exp(-_g0[_m:2 * _m]))
_gg0 = np.tanh(_g0[2 * _m:3 * _m]); _o0 = 1 / (1 + np.exp(-_g0[3 * _m:]))
_c1 = _f0 * 0.0 + _i0 * _gg0
assert np.allclose(np.asarray(_cell_eq.cache['c'][1]), _c1, atol=1e-12), "gate order f,i,g,o"
assert np.allclose(h_all_eq[0].detach().numpy(), _o0 * np.tanh(_c1), atol=1e-12)

# Autograd agrees with a central finite difference along b (all-ones direction).
_eps = 1e-6
with torch.no_grad():
    _cell_eq.b += _eps
_lp = float((_cell_eq.forward(X_eq) * torch.as_tensor(_dh_eq)).sum())
with torch.no_grad():
    _cell_eq.b -= 2 * _eps
_lm = float((_cell_eq.forward(X_eq) * torch.as_tensor(_dh_eq)).sum())
with torch.no_grad():
    _cell_eq.b += _eps
assert abs((_lp - _lm) / (2 * _eps) - float(db.sum())) < 1e-5, \
    "autograd BPTT must match finite differences"


### library

`nn.LSTMCell` computes identical gates but packs its rows i,f,g,o where the notebook stacks f,i,g,o — the classic silent bug when porting RNN weights between codebases. **What the library adds:** the fused kernel, plus that layout lesson: one row permutation (its own inverse) carries weights out and gradients back.

In [ ]:
import numpy as np
import torch

# hints:
# 1. torch's gate order is i,f,g,o — the notebook stacks f,i,g,o. Permute the rows.
# 2. Swapping the f and i blocks is its own inverse: one permutation maps both ways.
# 3. Scratch W @ [h, x] splits by columns: W[:, :m] is weight_hh, W[:, m:] is weight_ih.
# 4. Put the whole scratch bias into bias_ih (permuted) and zero bias_hh.
# 5. Translate gradients back the same way: dW = cat([hh.grad, ih.grad], 1)[perm].


class LSTMCell:
    """nn.LSTMCell fed the scratch weights. The entire content of this lane is
    the layout translation: torch packs gates i,f,g,o while the notebook stacks
    f,i,g,o, and the scratch W multiplies [h, x] so its columns split into the
    module's weight_hh | weight_ih."""

    def __init__(self, input_dim, hidden_dim, rng):
        self.hidden_dim = hidden_dim
        m = hidden_dim
        concat_dim = hidden_dim + input_dim
        scale = np.sqrt(2.0 / concat_dim)
        self.W = torch.as_tensor(rng.normal(0, scale, (4 * m, concat_dim)))
        b = np.zeros(4 * m)
        b[:m] = 1.0
        self.b = torch.as_tensor(b)
        # f,i,g,o -> i,f,g,o is a swap of the first two blocks: an involution,
        # so the same index array converts weights out and gradients back.
        self._perm = np.concatenate([np.arange(m, 2 * m), np.arange(0, m),
                                     np.arange(2 * m, 3 * m), np.arange(3 * m, 4 * m)])
        self.cell = torch.nn.LSTMCell(input_dim, m).double()
        with torch.no_grad():
            self.cell.weight_hh.copy_(self.W[self._perm, :m])
            self.cell.weight_ih.copy_(self.W[self._perm, m:])
            self.cell.bias_ih.copy_(self.b[self._perm])
            self.cell.bias_hh.zero_()

    def forward(self, X):
        """One nn.LSTMCell call per step (batch of 1), caching h and c."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        m = self.hidden_dim
        h = torch.zeros(1, m, dtype=torch.float64)
        c = torch.zeros(1, m, dtype=torch.float64)
        self.cache = {'h': [h[0].detach().clone()], 'c': [c[0].detach().clone()]}
        h_list = []
        for t in range(Xt.shape[0]):
            h, c = self.cell(Xt[t].unsqueeze(0), (h, c))
            self.cache['h'].append(h[0].detach().clone())
            self.cache['c'].append(c[0].detach().clone())
            h_list.append(h[0])
        self.h_all_ = torch.stack(h_list)
        return self.h_all_

    def backward(self, dh_all):
        """Autograd through the module, then permute the grads back to f,i,g,o."""
        dh = torch.as_tensor(np.asarray(dh_all, dtype=float))
        self.cell.zero_grad(set_to_none=True)
        (self.h_all_ * dh).sum().backward()
        dW_torch = torch.cat([self.cell.weight_hh.grad, self.cell.weight_ih.grad], dim=1)
        self.dW = dW_torch[self._perm].detach()
        self.db = self.cell.bias_ih.grad[self._perm].detach()


In [ ]:
# exports: h_last, c_last, dW, db
_rng_eq = np.random.default_rng(716)
_cell_eq = LSTMCell(input_dim=3, hidden_dim=5, rng=_rng_eq)
X_eq = _rng_eq.normal(size=(8, 3))
h_all_eq = _cell_eq.forward(X_eq)
_dh_eq = np.linspace(-1.0, 1.0, 40).reshape(8, 5)
_cell_eq.backward(_dh_eq)
h_last = h_all_eq[-1].detach()
c_last = _cell_eq.cache['c'][-1]
dW = _cell_eq.dW
db = _cell_eq.db
print("h_last:", np.round(h_last.numpy(), 5))
print("|dW| =", float(torch.linalg.norm(dW)))


In [ ]:
# In torch layout the forget block is SECOND, so the bias of 1.0 lands there.
assert torch.all(_cell_eq.cell.bias_ih.detach()[5:10] == 1.0), "forget bias moved to block 2"
assert float(_cell_eq.cell.bias_ih.detach()[:5].abs().max()) == 0.0, "input-gate bias stays 0"

# Round trip: permuting torch's rows back reproduces the scratch stack exactly.
_Wt = torch.cat([_cell_eq.cell.weight_hh.detach(), _cell_eq.cell.weight_ih.detach()], dim=1)
assert torch.equal(_Wt[_cell_eq._perm], _cell_eq.W), "the f/i swap is its own inverse"

# The split bias is redundant here too: both halves get the identical gradient.
assert torch.allclose(_cell_eq.cell.bias_ih.grad, _cell_eq.cell.bias_hh.grad, atol=1e-12)
